# Checkpoint 5: Testing hypotheses before parametric testing
I need to check two assumptions those tests rely on: normality (is the data roughly bell-shaped?) and equal variance (do the groups spread out by a similar amount?). Here I check both, for both tests.

## 1. Load df_clean

In [125]:
import pandas as pd
import numpy as np
from scipy import stats


In [126]:
cols = [
    "id_x", "car_rel_url_x", "datetime_scrape", "price_x", "currency_x", "city",
    "production_year", "engine_displacement_num", "kilometrage_num", "Marka", "Model",
    "Sürətlər qutusu", "Vəziyyəti", "Ötürücü", "Ban növü", "views"
]

In [127]:
df = pd.read_csv("cars.csv", usecols = cols, parse_dates = ["datetime_scrape"])

In [128]:
df

,id_x,car_rel_url_x,datetime_scrape,price_x,currency_x,city,production_year,engine_displacement_num,kilometrage_num,views,Ban növü,Marka,Model,Sürətlər qutusu,Vəziyyəti,Ötürücü
0,3c234145-d57a-4ad6-9448-d43810fc3392,/autos/8748840-hyundai-i30,2024-09-13 20:32:19.751157+00,15000.0,AZN,bakı,2008,1.6,270000,492,"Hetçbek, 5 qapı",Hyundai,i30,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Ön
1,c74ea36f-6be1-4de4-926d-e117197dcf00,/autos/8475807-lada-vaz-niva-travel,2024-09-13 20:32:19.751157+00,23700.0,AZN,bakı,2024,1.7,0,60189,"Offroader / SUV, 5 qapı",LADA (VAZ),Niva Travel,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
2,9cefceb0-024d-4581-a869-a3c2c68a9f95,/autos/8739686-toyota-land-cruiser,2024-09-13 20:32:19.751157+00,35600.0,$,bakı,2011,4.0,164750,2473,"Offroader / SUV, 5 qapı",Toyota,Land Cruiser,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam
3,459cc337-fb63-48de-9694-41554923d311,/autos/8712597-hyundai-elantra,2024-09-13 20:32:19.751157+00,26700.0,AZN,bakı,2018,2.0,126000,3727,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
4,6c5ee8d8-1c6f-4fad-a694-957a4c43c25d,/autos/8674773-toyota-prius,2024-09-13 20:32:19.751157+00,10500.0,AZN,bakı,2007,1.5,354000,446,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653716,16caa803-a546-455b-81ff-bc20868c2136,/autos/9081944-toyota-prius,2025-01-05 20:15:21.051803,10800.0,AZN,bakı,2008,1.5,320000,210,Liftbek,Toyota,Prius,Variator,"Vuruğu yoxdur, rənglənməyib",Ön
653717,2a26a6c1-8914-4b68-abb4-1fbd12ced7bc,/autos/9081939-uaz-hunter,2025-01-05 20:15:21.051803,9500.0,AZN,göygöl,2011,2.9,155000,1195,"Offroader / SUV, 5 qapı",UAZ,Hunter,Mexaniki,"Vuruğu yoxdur, rənglənməyib",Tam
653718,feb75615-0131-4d6f-b009-cc02faea4e01,/autos/9055034-hyundai-elantra,2025-01-05 20:15:21.051803,25400.0,AZN,bakı,2018,2.0,77926,1120,Sedan,Hyundai,Elantra,Avtomat,"Vuruğu yoxdur, rənglənməyib",Ön
653719,e5f8e957-5543-42be-b30f-78723ce551f9,/autos/9065903-jeep-grand-cherokee,2025-01-05 20:15:21.051803,10600.0,AZN,kürdəmir,1999,4.7,250000,1320,"Offroader / SUV, 5 qapı",Jeep,Grand Cherokee,Avtomat,"Vuruğu yoxdur, rənglənməyib",Tam


In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 653721 entries, 0 to 653720
Data columns (total 16 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id_x                     653721 non-null  object 
 1   car_rel_url_x            653721 non-null  object 
 2   datetime_scrape          653721 non-null  object 
 3   price_x                  653721 non-null  float64
 4   currency_x               653721 non-null  object 
 5   city                     653721 non-null  object 
 6   production_year          653721 non-null  int64  
 7   engine_displacement_num  653721 non-null  float64
 8   kilometrage_num          653721 non-null  int64  
 9   views                    653721 non-null  int64  
 10  Ban növü                 653721 non-null  object 
 11  Marka                    653721 non-null  object 
 12  Model                    653720 non-null  object 
 13  Sürətlər qutusu          653721 non-null  object 
 14  Vəzi

In [130]:
df_dedup = df.sort_values("datetime_scrape").drop_duplicates(subset = "car_rel_url_x", keep = "last").copy()

In [131]:
exchange_rate = {"AZN": 1.0, "$": 1.70, "€": 1.85}
df_dedup["price_azn"] = df_dedup["price_x"] * df_dedup["currency_x"].map(exchange_rate)


In [132]:
exclude_body_types = ["Yük maşını", "Motosiklet", "Avtobus", "Moped", "Kvadrosikl", "Dartqı", "Mikroavtobus"]
df_clean = df_dedup[~df_dedup["Ban növü"].isin(exclude_body_types)].copy()
df_clean = df_clean[df_clean["price_azn"] >= 1000].copy()

In [133]:
manual = df_clean.loc[df_clean["Sürətlər qutusu"] == "Mexaniki", "price_azn"]
auto = df_clean.loc[df_clean["Sürətlər qutusu"] == "Avtomat", "price_azn"]
print("n manual:", len(manual), "| n auto:", len(auto))

n manual: 37161 | n auto: 98161


In [134]:
df_clean.shape

(149478, 17)

## 2. Normality check - gearbox groups (Automatic vs. Manual)

In [135]:
for name, grp in [("manual", manual), ("auto", auto)]:
    skew = stats.skew(grp)
    stat, p = stats.normaltest(grp)  # D'Agostino K^2, works fine for large n (Shapiro doesn't)
    print(f"{name}: skew = {skew:.2f}, normality test p = {p:.2e}")

manual: skew = 14.31, normality test p = 0.00e+00
auto: skew = 6.28, normality test p = 0.00e+00


both groups fail the normality test, and skew is very high (14.3 for manual, 6.3 for auto) instead of close to 0. Price data has a long right tail - most cars are cheap, a few are extremely expensive, so it's not bell-shaped. Normality assumption doesn't hold.

## 3. Equal variance check - gearbox groups

In [136]:
lev_stat, lev_p = stats.levene(auto, manual)
print(f"Levene's test: stat = {lev_stat:.1f}, p = {lev_p:.2e}")
print(f"std auto: {auto.std():.0f} | std manual: {manual.std():.0f}")

Levene's test: stat = 4751.0, p = 0.00e+00
std auto: 34053 | std manual: 8188


Levene's test also fails. The two groups don't have equal variance. Automatic car prices spread out much more than manual car prices, which makes sense since automatics include everything from cheap to very expensive imports.

I already used Welch's t-test in Checkpoint 3, which doesn't assume equal variance, so that part is already handled. Normality is more of a concern, but with sample sizes this large, the Central Limit Theorem means the sampling distribution of the mean is still close to normal even though the raw prices aren't.

In [137]:
u_stat, u_p = stats.mannwhitneyu(auto, manual, alternative='two-sided')
print(f"Mann-Whitney U: p = {u_p:.2e}")

Mann-Whitney U: p = 0.00e+00


Mann-Whitney U also gives p = 0, same conclusion as the Welch's t-test. So the original result holds up even without the normality assumption.

## 4. Normality check - brand groups (for ANOVA)

In [138]:
top5_brands = ["Mercedes", "Hyundai", "Kia", "Toyota", "LADA (VAZ)"]
groups = [df_clean.loc[df_clean["Marka"] == b, "price_azn"] for b in top5_brands]

In [139]:
for b, g in zip(top5_brands, groups):
    skew = stats.skew(g)
    stat, p = stats.normaltest(g)
    print(f"{b}: n = {len(g)}, skew = {skew:.2f}, p = {p:.2e}")

Mercedes: n = 25307, skew = 5.97, p = 0.00e+00
Hyundai: n = 19658, skew = 2.03, p = 0.00e+00
Kia: n = 14619, skew = 1.08, p = 0.00e+00
Toyota: n = 14388, skew = 2.38, p = 0.00e+00
LADA (VAZ): n = 12859, skew = 2.20, p = 0.00e+00


all 5 brand groups fail normality too, same reason - right-skewed prices, some more than others

Mercedes is the most skewed since it has the widest price range, from cheap old models to very expensive new ones.

## 5. Equal variance check - brand groups (for ANOVA)

In [140]:
lev_stat2, lev_p2 = stats.levene(*groups)
print(f"Levene's test (5 brands): stat = {lev_stat2:.1f}, p = {lev_p2:.2e}")
for b, g in zip(top5_brands, groups):
    print(f"{b}: std = {g.std():.0f}")

Levene's test (5 brands): stat = 1188.2, p = 0.00e+00
Mercedes: std = 37616
Hyundai: std = 9818
Kia: std = 11731
Toyota: std = 22717
LADA (VAZ): std = 4728


variances are very unequal across brands too. Mercedes spreads out 8x more than LADA (37,616 vs. 4,728). So standard ANOVA's equal-variance assumption is violated as well. As a check, I ran Kruskal-Wallis, the non-parametric version of ANOVA that doesn't need normal or equal-variance groups:

In [141]:
kw_stat, kw_p = stats.kruskal(*groups)
print(f"Kruskal-Wallis: stat = {kw_stat:.1f}, p = {kw_p:.2e}")

Kruskal-Wallis: stat = 28426.1, p = 0.00e+00


Kruskal-Wallis also gives p = 0, matching the original ANOVA conclusion. So even though both assumptions fail, the "at least one brand differs" result still holds.

Neither test's assumptions actually held: prices are right-skewed and the groups don't have equal variance, in both the gearbox comparison and the brand comparison. But since I already used Welch's t-test and confirmed both results with non-parametric tests that don't need those assumptions, the original conclusions from Checkpoint 3 still stand.